# GEE NASA GPM IMERG Downloader (v07 Final Run)
Notebook ini didesain khusus dan modular untuk mengunduh produk presipitasi satelit global standar emas dunia **NASA GPM IMERG v07 (Integrated Multi-satellitE Retrievals for GPM)** dari Google Earth Engine (GEE):

### Koleksi & Spesifikasi:
1. **IMERG Bulanan Final Run:** `NASA/GPM_L3/IMERG_MONTHLY_V07`
   - **Variabel Utama:** `precipitation` (laju rerata mm/hari $\rightarrow$ diakumulasi ke total mm/bulan)
   - **Variabel Kualitas & Kalibrasi:** `gaugeRelativeWeighting`, `precipitationQualityIndex`, `randomError`
   - **Resolusi Spasial:** $0.10^\circ \times 0.10^\circ$ ($\sim 11.1\text{ km}$)
   - **Cakupan Temporal:** Januari 1998 s.d. Sekarang (Era TRMM hingga GPM)
2. **IMERG Setengah Jam (Half-Hourly):** `NASA/GPM_L3/IMERG_V07`
   - **Variabel:** `precipitation` (mm/jam), `MWprecipitation` (PMW murni), `IRprecipitation` (IR murni)
   - **Mode Agregasi:** Agregasi Harian (Daily Accumulation) per bulan (28--31 band $\le 1024$ band)

### Fitur Utama Sesuai Project Rules:
- **Dual Environment Compatibility:** Otomatis mendeteksi eksekusi di **Lokal** (via Service Account JSON) maupun **Kaggle Notebooks** (via `KAGGLE_KERNEL_RUN_TYPE` & `kaggle_secrets`).
- **Vector Boundary & BBox Safety Buffer (Rule 1):** Ekstraksi poligon `33.05_kecamatan.geojson` dengan pembersihan 2D dan penambahan buffer $\pm 0.01^\circ$.
- **Multi-Band Stacking (`toBands()` - Rule 3):** Mempercepat throughput API hingga 30x lipat tanpa *rate-limit*.
- **Hierarchical Data Architecture (Rule 3):** Disimpan rapi pada struktur `data/imerg_monthly/<tahun>/imerg_monthly_<tahun>_<bulan>.nc`.
- **Visual Quick-Check:** Menampilkan peta spasial klimatologi Kebumen dan perbandingan deret waktu.

In [ ]:
# =========================================================================
# 0. INSTALASI DEPENDENSI (JIKA DI RUN DI KAGGLE / ENVIRONMENT BARU)
# =========================================================================
!pip install geemap earthengine-api rioxarray geopandas xarray netCDF4 matplotlib --quiet

In [ ]:
# =========================================================================
# 1. IMPORT MODUL & PENGATURAN DIREKTORI DINAMIS
# =========================================================================
import ee
import geopandas as gpd
import geemap
from shapely.validation import make_valid
from shapely.ops import transform, unary_union
import os
import shutil
import glob
import rioxarray as rxr
import xarray as xr
import pandas as pd
import numpy as np
import calendar
from datetime import datetime, timedelta
import matplotlib.pyplot as plt

BASE_DIR = os.getcwd()

# Deteksi Lokasi File GeoJSON Administrasi Kebumen
if 'KAGGLE_KERNEL_RUN_TYPE' in os.environ:
    file_geojson = "/kaggle/input/datasets/jerismeteo/projek-downscale/33.05_kecamatan.geojson"
    if not os.path.exists(file_geojson):
        found_geo = glob.glob("/kaggle/input/**/33.05_kecamatan.geojson", recursive=True)
        file_geojson = found_geo[0] if found_geo else os.path.join(BASE_DIR, "33.05_kecamatan.geojson")
else:
    file_geojson = os.path.join(BASE_DIR, "33.05_kecamatan.geojson")

# Struktur Folder Hierarkis (Rule 3)
FOLDER_IMERG_MONTHLY = os.path.join(BASE_DIR, "data", "imerg_monthly")
FOLDER_IMERG_DAILY = os.path.join(BASE_DIR, "data", "imerg_daily")

# Lokasi Smart Cache jika data sudah tersedia di Input Dataset Kaggle
KAGGLE_IMERG_INPUT = "/kaggle/input/datasets/jerismeteo/gee-imerg-kebumen/data"

TAHUN_AWAL = 2000
TAHUN_AKHIR = 2025

print(f"📌 File Batas Wilayah GeoJSON : {file_geojson}")
print(f"📌 Direktori Output IMERG Bln : {FOLDER_IMERG_MONTHLY}")
print(f"📌 Direktori Output IMERG Hrn : {FOLDER_IMERG_DAILY}")
print(f"📌 Target Rentang Tahun       : {TAHUN_AWAL} s.d. {TAHUN_AKHIR}")

In [ ]:
# =========================================================================
# 2. INISIALISASI GOOGLE EARTH ENGINE (GEE)
# Mendukung: 1. Service Account Lokal (.json) | 2. Kaggle Secrets (GEE_KEY) | 3. Fallback Auth
# =========================================================================
import os
import json
import ee
from google.oauth2.service_account import Credentials

PROJECT_ID = 'staklimjerukagung'
init_success = False

# 1. Coba via Service Account File Lokal (Repo Projek_Downscale)
sa_candidates = [
    os.path.join(BASE_DIR, 'staklimjerukagung-b852a12a367e.json'),
    os.path.join(os.path.dirname(BASE_DIR), 'staklimjerukagung-b852a12a367e.json'),
    'staklimjerukagung-b852a12a367e.json'
]
for sa_path in sa_candidates:
    if os.path.exists(sa_path):
        try:
            with open(sa_path, 'r') as f:
                sa_info = json.load(f)
            SCOPES = ['https://www.googleapis.com/auth/earthengine']
            creds = Credentials.from_service_account_info(sa_info, scopes=SCOPES)
            ee.Initialize(credentials=creds, project=sa_info.get('project_id', PROJECT_ID))
            print(f"✅ Berhasil Inisialisasi GEE via File Service Account Lokal: {os.path.basename(sa_path)}")
            init_success = True
            break
        except Exception as e:
            print(f"⚠️ Gagal inisialisasi via SA file {sa_path}: {e}")

# 2. Coba via Kaggle User Secrets (Kaggle Environment: Add-ons -> Secrets -> GEE_KEY)
if not init_success:
    try:
        from kaggle_secrets import UserSecretsClient
        user_secrets = UserSecretsClient()
        sa_json_str = user_secrets.get_secret('GEE_KEY')
        sa_info = json.loads(sa_json_str)
        SCOPES = ['https://www.googleapis.com/auth/earthengine']
        creds = Credentials.from_service_account_info(sa_info, scopes=SCOPES)
        ee.Initialize(credentials=creds, project=sa_info.get('project_id', PROJECT_ID))
        print("✅ Berhasil Inisialisasi GEE via Kaggle Secret (GEE_KEY)")
        init_success = True
    except Exception as e:
        print(f"⚠️ Info Kaggle Secrets: {e}")
        print("💡 Tips di Kaggle: Buka menu Add-ons -> Secrets -> pastikan centang (checkbox) pada 'GEE_KEY' di notebook ini!")

# 3. Fallback ke Inisialisasi Default / Browser OAuth
if not init_success:
    try:
        ee.Initialize(project=PROJECT_ID)
        print("✅ Berhasil Inisialisasi GEE via Session Default")
    except Exception as e:
        print(f"ℹ️ Inisialisasi default gagal ({e}), membuka otentikasi browser GEE...")
        ee.Authenticate()
        ee.Initialize(project=PROJECT_ID)
        print("✅ Berhasil Autentikasi & Inisialisasi GEE via Browser")


In [ ]:
# =========================================================================
# 3. PERSIAPAN GEOMETRI & BOUNDING BOX (RULE 1)
# =========================================================================
if not os.path.exists(file_geojson):
    raise FileNotFoundError(f"File GeoJSON tidak ditemukan: {file_geojson}")

gdf = gpd.read_file(file_geojson)
if gdf.crs != "EPSG:4326":
    gdf = gdf.to_crs("EPSG:4326")

print("Membersihkan topologi geometri...")
gdf = gdf[gdf.geometry.notna()].copy()

def _to_2d(geom):
    if geom is None or geom.is_empty: return None
    return transform(lambda x, y, z=None: (x, y), geom)

def _clean_geom(geom):
    if geom is None or geom.is_empty: return None
    geom = _to_2d(geom)
    geom = make_valid(geom)
    if geom is None or geom.is_empty: return None
    return geom.buffer(0)

gdf["geometry"] = gdf["geometry"].apply(_clean_geom)
gdf = gdf[gdf.geometry.notna() & ~gdf.geometry.is_empty].copy()
gdf = gdf[gdf.geom_type.isin(["Polygon", "MultiPolygon"])].copy()
gdf.reset_index(drop=True, inplace=True)

print("Mengonversi ke Google Earth Engine FeatureCollection...")
geojson_fc = gdf.__geo_interface__
batas_kebumen = ee.FeatureCollection(geojson_fc["features"])

# Ekstraksi BBox dengan Safety Buffer (Rule 1)
bounds = gdf.total_bounds
BUFFER_DEG = 0.02
ee_bbox = ee.Geometry.BBox(
    bounds[0] - BUFFER_DEG,  # min Lon (West)
    bounds[1] - BUFFER_DEG,  # min Lat (South)
    bounds[2] + BUFFER_DEG,  # max Lon (East)
    bounds[3] + BUFFER_DEG   # max Lat (North)
)

print(f"✅ Batas Geometri Siap!")
print(f"   Extent Asli: Lon [{bounds[0]:.4f}, {bounds[2]:.4f}], Lat [{bounds[1]:.4f}, {bounds[3]:.4f}]")
print(f"   Extent BBox: Lon [{bounds[0]-BUFFER_DEG:.4f}, {bounds[2]+BUFFER_DEG:.4f}], Lat [{bounds[1]-BUFFER_DEG:.4f}, {bounds[3]+BUFFER_DEG:.4f}]")

In [ ]:
# =========================================================================
# 4. FUNGSI UNDUH SUPER-CEPAT GPM IMERG BULANAN (FINAL RUN V07)
# =========================================================================
def unduh_imerg_bulanan(tahun, output_base_dir):
    """
    Mengunduh 12 bulan presipitasi bulanan NASA GPM IMERG v07 Final Run
    menggunakan teknik Multi-Band Stacking (toBands()) dalam satu kali request.
    """
    folder_tahun = os.path.join(output_base_dir, str(tahun))
    os.makedirs(folder_tahun, exist_ok=True)
    
    nc_out_path = os.path.join(folder_tahun, f"imerg_monthly_{tahun}.nc")
    tif_temp_path = os.path.join(folder_tahun, f"imerg_monthly_{tahun}_temp.tif")
    
    if os.path.exists(nc_out_path):
        print(f"[{tahun}] ✓ File NetCDF tahunan sudah ada: {nc_out_path}")
        return nc_out_path
        
    tgl_awal = f"{tahun}-01-01"
    tgl_akhir = f"{tahun+1}-01-01"
    
    print(f"[{tahun}] Mengambil koleksi IMERG Monthly V07 dari GEE...")
    col = (ee.ImageCollection("NASA/GPM_L3/IMERG_MONTHLY_V07")
           .filterBounds(ee_bbox)
           .filterDate(tgl_awal, tgl_akhir))
    
    n_img = col.size().getInfo()
    if n_img == 0:
        print(f"[{tahun}] ⚠️ Tidak ada data IMERG di GEE untuk tahun ini.")
        return None
        
    # Ambil timestamps
    times_ms = col.aggregate_array('system:time_start').getInfo()
    time_index = pd.to_datetime(times_ms, unit='ms')
    
    # Seleksi band presipitasi (dalam mm/hari) dan konversi ke mm/bulan (x jumlah hari)
    def convert_to_monthly_accum(img):
        # Ambil tanggal untuk menghitung jumlah hari dalam bulan tersebut
        date = ee.Date(img.get('system:time_start'))
        # days in month = date.advance(1, 'month').difference(date, 'day')
        days = date.advance(1, 'month').difference(date, 'day')
        rate_mm_day = img.select('precipitation')
        monthly_mm = rate_mm_day.multiply(days).rename('precipitation_monthly_mm')
        return monthly_mm.copyProperties(img, ['system:time_start'])
        
    monthly_col = col.map(convert_to_monthly_accum)
    stacked = monthly_col.toBands().clip(ee_bbox)
    
    print(f"[{tahun}] Mengekspor {n_img} bulan sekaligus ke GeoTIFF temporer...")
    geemap.ee_export_image(
        stacked,
        filename=tif_temp_path,
        region=ee_bbox,
        scale=11132,  # Resolusi asli ~0.10 derajat (11 km)
        file_per_band=False
    )
    
    print(f"[{tahun}] Mengonversi GeoTIFF -> NetCDF Time-Series...")
    with rxr.open_rasterio(tif_temp_path, masked=True) as da:
        da = da.rename({'band': 'time'})
        da['time'] = time_index[:len(da.time)]
        da.name = "precipitation"
        da.attrs["long_name"] = "NASA GPM IMERG v07 Final Run Monthly Accumulated Precipitation"
        da.attrs["units"] = "mm/month"
        da.attrs["source"] = "NASA/GPM_L3/IMERG_MONTHLY_V07"
        da.to_netcdf(nc_out_path)
        
    if os.path.exists(tif_temp_path): os.remove(tif_temp_path)
    print(f"[{tahun}] ✓ Sukses! Tersimpan di: {nc_out_path}\n")
    return nc_out_path

In [ ]:
# =========================================================================
# 5. FUNGSI UNDUH IMERG HARIAN AGREGASI (DAILY ACCUMULATION V07)
# =========================================================================
def unduh_imerg_harian_bulanan(tahun, bulan, output_base_dir):
    """
    Mengunduh presipitasi harian (mm/hari) selama 1 bulan penuh
    diagregasikan dari koleksi 30-menit NASA/GPM_L3/IMERG_V07.
    Menggunakan 28-31 band per bulan (jauh di bawah limit 1024 band GEE).
    """
    folder_tahun = os.path.join(output_base_dir, str(tahun))
    os.makedirs(folder_tahun, exist_ok=True)
    
    nc_path = os.path.join(folder_tahun, f"imerg_daily_{tahun}_{bulan:02d}.nc")
    tif_temp = os.path.join(folder_tahun, f"imerg_daily_{tahun}_{bulan:02d}_temp.tif")
    
    if os.path.exists(nc_path):
        print(f"[{tahun}-{bulan:02d}] ✓ File harian sudah ada: {nc_path}")
        return nc_path
        
    num_days = calendar.monthrange(tahun, bulan)[1]
    daily_images = []
    dates_list = []
    
    print(f"[{tahun}-{bulan:02d}] Mengompilasi agregasi harian ({num_days} hari) dari citra 30-menit...")
    for day in range(1, num_days + 1):
        t_start = f"{tahun}-{bulan:02d}-{day:02d}"
        if day == num_days:
            if bulan == 12:
                t_end = f"{tahun+1}-01-01"
            else:
                t_end = f"{tahun}-{bulan+1:02d}-01"
        else:
            t_end = f"{tahun}-{bulan:02d}-{day+1:02d}"
            
        sub_col = (ee.ImageCollection("NASA/GPM_L3/IMERG_V07")
                   .filterBounds(ee_bbox)
                   .filterDate(t_start, t_end)
                   .select('precipitation'))
        
        # Laju curah hujan rata-rata (mm/jam) * 24 jam = total presipitasi harian (mm/hari)
        daily_img = sub_col.mean().multiply(24.0).rename(f"p_day_{day:02d}")
        daily_images.append(daily_img)
        dates_list.append(pd.to_datetime(t_start))
        
    # Stack 28-31 band harian menjadi 1 citra multi-band
    stacked_daily = ee.ImageCollection(daily_images).toBands().clip(ee_bbox)
    
    print(f"[{tahun}-{bulan:02d}] Mengekspor citra multi-band harian ke TIF...")
    geemap.ee_export_image(
        stacked_daily,
        filename=tif_temp,
        region=ee_bbox,
        scale=11132,
        file_per_band=False
    )
    
    print(f"[{tahun}-{bulan:02d}] Menyusun NetCDF Harian Time-Series...")
    with rxr.open_rasterio(tif_temp, masked=True) as da:
        da = da.rename({'band': 'time'})
        da['time'] = dates_list[:len(da.time)]
        da.name = "precipitation"
        da.attrs["long_name"] = "Daily Precipitation (Aggregated from 30-min GPM IMERG v07)"
        da.attrs["units"] = "mm/day"
        da.to_netcdf(nc_path)
        
    if os.path.exists(tif_temp): os.remove(tif_temp)
    print(f"[{tahun}-{bulan:02d}] ✓ Selesai! NetCDF Harian: {nc_path}\n")
    return nc_path

In [ ]:
# =========================================================================
# 6. EKSEKUSI BATCH PENGUNDUHAN MULTI-TAHUN IMERG BULANAN (2000 - 2025)
# =========================================================================
print("=" * 75)
print(f"🚀 MEMULAI PIPELINE BATCH PENGUNDUHAN GPM IMERG BULANAN ({TAHUN_AWAL} - {TAHUN_AKHIR})")
print("=" * 75)

downloaded_files = []
for thn in range(TAHUN_AWAL, TAHUN_AKHIR + 1):
    path_res = unduh_imerg_bulanan(thn, FOLDER_IMERG_MONTHLY)
    if path_res:
        downloaded_files.append(path_res)

print("=" * 75)
print(f"🎉 SELESAI PENGUNDUHAN BATCH: {len(downloaded_files)} TAHUN FILE NETCDF TERSEDIA!")
print("=" * 75)

In [ ]:
# =========================================================================
# 7. VERIFIKASI VISUALISASI SPASIAL KLIMATOLOGI IMERG KEBUMEN
# =========================================================================
list_nc = sorted(glob.glob(os.path.join(FOLDER_IMERG_MONTHLY, "*", "*.nc")))

if not list_nc:
    print("ℹ️ Belum ada file NetCDF IMERG yang selesai diunduh untuk divisualisasikan.")
else:
    print(f"Memuat {len(list_nc)} file tahunan NetCDF untuk kalkulasi klimatologi...")
    datasets = [xr.open_dataset(f) for f in list_nc]
    ds_all = xr.concat(datasets, dim="time")
    
    # Total akumulasi tahunan rata-rata (mm/tahun)
    annual_mean = ds_all["precipitation"].resample(time="1YS").sum(dim="time").mean(dim="time")
    
    fig, ax = plt.subplots(figsize=(10, 6), dpi=150)
    p = annual_mean.plot(
        ax=ax,
        cmap="Blues",
        cbar_kwargs={"label": "Presipitasi Tahunan Rerata (mm/tahun)"}
    )
    gdf.boundary.plot(ax=ax, color="black", linewidth=0.8, alpha=0.8)
    ax.set_title(f"Klimatologi Spasial Presipitasi NASA GPM IMERG v07 ({TAHUN_AWAL}--{TAHUN_AKHIR})\nKabupaten Kebumen, Jawa Tengah", fontsize=11, fontweight="bold")
    ax.set_xlabel("Bujur (Longitude)")
    ax.set_ylabel("Lintang (Latitude)")
    ax.grid(True, linestyle="--", alpha=0.5)
    plt.tight_layout()
    plt.show()
    
    # Tutup dataset untuk hemat RAM
    ds_all.close()
    for ds in datasets: ds.close()